# Instructions
1. So you will read in the photometry for a galaxy, 
2. pull the spectrum from Blast, 
3. compute the k correction, 
4. and add (or subtract depending on convention) to the photometry 

### goal : to make corrected photometry.

In [ ]:
#remake those og graphes with the corrected fluxes

In [2]:
import numpy as np
import pandas as pd
import  matplotlib.pyplot as plt
import math
from urllib.request import urlopen
import json

In [ ]:
#step 1 - for sn2022hrs ==. tbd

In [3]:
def safe_get(data, key, default=None):
    return data.get(key, default)

In [4]:
#step 2- going to assume im getting model spectrum, ie SED

#nvm it looks like : https://blast.scimma.org/download_modelfit/SN2017erp/global
# with the URL path `<base_blast_url>/download_modelfit/<transient_name>/<aperture_type>`
#model downloaded in units of maggies - YAY

SNList = ['SN2017erp']

for name in SNList:
    url = f"https://blast.scimma.org/download_modelfit/{name}/global"
    response = urlopen(url)
    
    #saving it
    folder = 'model_datas'
    filename = f"{folder}/{name}_global_modeldata.npz"
    with open(filename, "wb") as f:
        f.write(response.read())
    #opens a file, writes in binary (wb) 
    

In [5]:
data = np.load('model_datas/SN2017erp_global_modeldata.npz')
#gives keys lipe 'phot' 
print(data.files)

wave = data['rest_wavelength']
flux = data['spec']
#for error
spec_84 = data['spec_84']
spec_16 = data['spec_16']

['rest_wavelength', 'spec', 'phot', 'spec_16', 'spec_84', 'phot_16', 'phot_84']


#### step 3 overview: k correction

To account for the stretching of the wavelength at a hypothetical z value, I multiplied all my wavelengths by (1+z).

To account for the stretching of the space, I divided all my flux by (1+z).

To account for time dilation, I multiplied my epochs by (1+z).

In [6]:
# to find z + reshifted sed 

for name in SNList:
    response = urlopen(f"https://blast.scimma.org/api/transient/get/{name}?format=json")
    data = json.loads(response.read())
    
    z = safe_get(data, "host_redshift")
    print(z)

wave_z = wave * (1+z)
flux_z = flux / (1+z)


0.006174


In [7]:
#K(z) = m_obs - m_rest
# --> finding mags
# m =22.5 - 2.5*log10(flux)

#removing 0, negs
good = (flux>0) & (flux_z>0)

flux_good = flux[good]
flux_z_good = flux_z[good]

#magnitudes
m_obs = 22.5 - 2.5* np.log10(flux_z)
m_rest = 22.5 - 2.5* np.log10(flux)

K = m_obs - m_rest

#save into df
#kcorr_df = pd.DataFrame({'wave_z': wave_z,'K': K})

<ipython-input-7-68dab0fbcc87>:12: RuntimeWarning: invalid value encountered in log10
  m_obs = 22.5 - 2.5* np.log10(flux_z)
<ipython-input-7-68dab0fbcc87>:13: RuntimeWarning: invalid value encountered in log10
  m_rest = 22.5 - 2.5* np.log10(flux)



Filter	Central Wavelength (Å)
- v	5468
- b	4392
- u	3465
- uvw1	2600
- uvm2	2246
- uvw2	1928

In [8]:
 filter_lambdas = {
    'UVW2': 1928,
    'UVM2': 2246,
    'UVW1': 2600,
    'U':    3465,
    'B':    4392,
    'V':    5468
}
    
def get_K_at_lambda (wave_array,k_array, target_lambda):
    idx = np.abs(wave_array - target_lambda).argmin()
    return k_array[idx]

In [9]:
K_values = {}

for filt, lam in filter_lambdas.items():
    K_values[filt] = get_K_at_lambda(wave_z, K, lam)

print(K_values)

{'UVW2': 0.006682726912757175, 'UVM2': 0.006682726912757175, 'UVW1': 0.006682726912757175, 'U': 0.006682726912757175, 'B': 0.00668272691275007, 'V': 0.006682726912757175}


In [13]:
#step 4 - find m_rest for observed photometry when you get it!
#doing it with what i have
#m_rest = m_obs - K(z)

#the phot tables
og_df = pd.read_csv('~/Desktop/aggie nova/galaxy project/eazy-photoz/FinalPhotUV.dat', sep  ='\s+')
og_df.set_index('sn', inplace=True)

columns = ['sn', 'K_UVW2', 'K_UVM2', 'K_UVW1', 'K_U', 'K_B', 'K_V']
final_rows = []


for name in SNList:
    row = {'sn': name}
    for filt, lam in K_values.items():
        m_obs = og_df.loc[name, filt]       # observed mag
        K_val = K_values[filt]              # K-correction value for this filter
        m_rest = m_obs - K_val              # rest-frame mag
        row[f'K_{filt}'] = m_rest
    #print(row)
        #m_final_rest = og_df.loc[name, filt] - lam
        #row[f'K_{filt}'] = m_final_rest
    final_rows.append(row)

final_df = pd.DataFrame(final_rows, columns=columns)
print(final_df.head())
print(final_df.shape) 
final_df.to_csv('trialKcorrectedFinalPhotUV.csv', index=False)

          sn     K_UVW2     K_UVM2     K_UVW1        K_U        K_B        K_V
0  SN2017erp  12.909573  12.980573  12.605573  12.266573  12.318573  11.593573
(1, 7)


In [11]:
#functions needed
def safe_get(data, key, default=None):
    return data.get(key, default)

filter_lambdas = {
    'UVW2': 1928,
    'UVM2': 2246,
    'UVW1': 2600,
    'U':    3465,
    'B':    4392,
    'V':    5468
}
    
def get_K_at_lambda (wave_array,k_array, target_lambda):
    idx = np.abs(wave_array - target_lambda).argmin()
    return k_array[idx]

In [12]:
#wait..
#combine everything into one huge large doc bae

#og phot data
og_df = pd.read_csv('~/Desktop/aggie nova/galaxy project/eazy-photoz/FinalPhotUV.dat', sep  ='\s+')
og_df.set_index('sn', inplace=True)

#final data setup
columns = ['sn','K_UVW2','Kerr_UVW2','K_UVM2','Kerr_UVM2','K_UVW1','Kerr_UVW1','K_U','Kerr_U','K_B','Kerr_B','K_V','Kerr_V']
final_rows = []

for name in og_df.index:
    #print(name)
    
    try:
        #getting model fit sed from blast
        url = f"https://blast.scimma.org/download_modelfit/{name}/global"
        response = urlopen(url)

        #saving it
        folder = 'model_datas'
        filename = f"{folder}/{name}_global_modeldata.npz"
        with open(filename, "wb") as f:
            f.write(response.read())

        #gathering relevant data
        data = np.load(f'model_datas/{name}_global_modeldata.npz')
        wave = data['rest_wavelength']
        flux = data['spec']
        spec_84 = data['spec_84']
        spec_16 = data['spec_16']

        # to find z + reshifted sed from blast
        response = urlopen(f"https://blast.scimma.org/api/transient/get/{name}?format=json")
        data = json.loads(response.read())

        z = safe_get(data, "host_redshift")
        wave_z = wave * (1+z)
        flux_z = flux / (1+z)

        #K(z) = m_obs - m_rest
        # m =22.5 - 2.5*log10(flux)

        #removing 0, negs
        good = (flux>0) & (flux_z>0)
        flux_good = flux[good]
        flux_z_good = flux_z[good]

        #convert from maggies to magnitudes
        m_obs = 22.5 - 2.5* np.log10(flux_z)
        m_rest = 22.5 - 2.5* np.log10(flux)

        #k value for ALL wavelngth
        K = m_obs - m_rest

        ##########
        #for error -tbd
        flux_84 = spec_84 / (1 + z)
        flux_16 = spec_16 / (1 + z)

        m_obs_84 = 22.5 - 2.5 * np.log10(flux_z)      # same obs mag
        m_rest_84 = 22.5 - 2.5 * np.log10(spec_84)
        m_rest_16 = 22.5 - 2.5 * np.log10(spec_16)

        K_84 = m_obs_84 - m_rest_84
        K_16 = m_obs_84 - m_rest_16

        K_err = 0.5 * np.abs(K_84 - K_16)
        ##########
        
        #k vals for uvot wavelength
        K_values = {}
        K_err_values = {}
        for filt, lam in filter_lambdas.items():
            K_values[filt] = get_K_at_lambda(wave_z, K, lam)
            K_err_values[filt] = get_K_at_lambda(wave_z, K_err, lam)

        #creating rows of final data
        row = {'sn': name}
        for filt, lam in K_values.items():
            m_obs = og_df.loc[name, filt]       # observed mag
            K_val = K_values[filt]              # K-correction value for this filter
            m_rest = m_obs - K_val              # rest-frame mag
            row[f'K_{filt}'] = m_rest
            row[f'Kerr_{filt}'] = K_err_values[filt]
        #adding em  
        final_rows.append(row)
        
    except Exception as e:
        print(f"{name}: model unavailable — {e}")
        row = {'sn': name}
        for filt in filter_lambdas:
            row[f'K_{filt}'] = np.nan
            row[f'Kerr_{filt}'] = np.nan
        #adding em  
        final_rows.append(row)    

#saving df        
final_df = pd.DataFrame(final_rows, columns=columns)
print(final_df.head())
print(final_df.shape) 
final_df.to_csv('KcorrectedFinalPhotUV.csv', index=False)


SN2016eii: model unavailable — HTTP Error 404: Not Found
SN2010ev: model unavailable — HTTP Error 404: Not Found


<ipython-input-12-566be0517169>:49: RuntimeWarning: invalid value encountered in log10
  m_obs = 22.5 - 2.5* np.log10(flux_z)
<ipython-input-12-566be0517169>:50: RuntimeWarning: invalid value encountered in log10
  m_rest = 22.5 - 2.5* np.log10(flux)
<ipython-input-12-566be0517169>:60: RuntimeWarning: invalid value encountered in log10
  m_obs_84 = 22.5 - 2.5 * np.log10(flux_z)      # same obs mag
<ipython-input-12-566be0517169>:61: RuntimeWarning: invalid value encountered in log10
  m_rest_84 = 22.5 - 2.5 * np.log10(spec_84)
<ipython-input-12-566be0517169>:62: RuntimeWarning: invalid value encountered in log10
  m_rest_16 = 22.5 - 2.5 * np.log10(spec_16)


SN2007bm: model unavailable — HTTP Error 404: Not Found
SN2012cp: model unavailable — HTTP Error 404: Not Found
SN2010ih: model unavailable — HTTP Error 404: Not Found
PS1-14xw: model unavailable — unsupported operand type(s) for +: 'int' and 'NoneType'
SN2012fr: model unavailable — HTTP Error 404: Not Found
SN2014J: model unavailable — HTTP Error 404: Not Found
SN2011de: model unavailable — unsupported operand type(s) for +: 'int' and 'NoneType'
SN2011at: model unavailable — HTTP Error 404: Not Found
SN2011dm: model unavailable — HTTP Error 404: Not Found
SN2013gh: model unavailable — HTTP Error 404: Not Found
SN2013cs: model unavailable — HTTP Error 404: Not Found
SN2008ha: model unavailable — unsupported operand type(s) for +: 'int' and 'NoneType'
SN2008Q: model unavailable — HTTP Error 404: Not Found
ASASSN-14lw: model unavailable — HTTP Error 404: Not Found
SN2013cg: model unavailable — HTTP Error 404: Not Found
SN2016blc: model unavailable — HTTP Error 404: Not Found
SN2009ig: mo